# cyp51A Gene Variants Analysis - v8.1 DEBUG

**DEBUG VERSION**: Investigating VCF content and fixing processing limits

**Issues to resolve**:
1. Remove artificial 1000-variant limit causing round numbers
2. Debug synonymous variant detection (0 filtered seems suspicious)
3. Examine actual VCF INFO field content
4. Implement proper variant effect analysis

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import gzip
import warnings
warnings.filterwarnings('ignore')
import gxy

plt.rcParams['figure.figsize'] = (16, 10)

print("🔍 DEBUG MODE: Investigating VCF processing issues")
print("🎯 Focus: Remove artificial limits and debug synonymous filtering")

## Step 1: Load GTF and Gene Coordinates (Same as before)

In [ ]:
# Quick GTF loading (reusing working code)
def _is_gzipped(path):
    with open(path, 'rb') as f:
        return f.read(2) == b'\x1f\x8b'

def extract_gene_info(attribute_string):
    if pd.isna(attribute_string): return None
    patterns = [(r'gene_id\s*["=]([^\s;";]+)', 'gene_id'), (r'(Afu\d+g\d+)', 'aspergillus_id')]
    for pattern, field in patterns:
        match = re.search(pattern, str(attribute_string), re.IGNORECASE)
        if match: return match.group(1)
    return None

# Load GTF
try:
    gtf_result = await gxy.get(4)
    gtf_file_path = gtf_result[0] if isinstance(gtf_result, list) else gtf_result
    is_compressed = _is_gzipped(gtf_file_path)
    opener = gzip.open if is_compressed else open
    gtf_columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
    
    with opener(gtf_file_path, 'rt' if is_compressed else 'r') as f:
        gtf_data = pd.read_csv(f, sep='\t', comment='#', names=gtf_columns, dtype=str)
    
    print(f"✓ GTF loaded: {len(gtf_data):,} annotations")
    gtf_loaded = True
except Exception as e:
    print(f"✗ GTF Error: {e}")
    gtf_loaded = False

# Find cyp51A gene
if gtf_loaded:
    gtf_data['gene_id'] = gtf_data['attribute'].apply(extract_gene_info)
    search_patterns = ['Afu4g06890', 'cyp51A', 'cyp51', 'CYP51']
    all_matches = []
    
    for pattern in search_patterns:
        matches = gtf_data[
            gtf_data['attribute'].str.contains(pattern, case=False, na=False) |
            gtf_data['gene_id'].str.contains(pattern, case=False, na=False)
        ]
        if len(matches) > 0: all_matches.append(matches)
    
    if all_matches:
        cyp51a_entries = pd.concat(all_matches, ignore_index=True).drop_duplicates()
        chromosome = cyp51a_entries.iloc[0]['seqname']
        cyp51a_entries['start'] = pd.to_numeric(cyp51a_entries['start'], errors='coerce')
        cyp51a_entries['end'] = pd.to_numeric(cyp51a_entries['end'], errors='coerce')
        gene_start = int(cyp51a_entries['start'].min())
        gene_end = int(cyp51a_entries['end'].max())
        
        # Analysis region
        roi_start = gene_start - 1000
        roi_end = gene_end + 1000
        
        print(f"✓ cyp51A: {chromosome}:{gene_start:,}-{gene_end:,}")
        print(f"✓ Analysis region: {chromosome}:{roi_start:,}-{roi_end:,}")
        gene_found = True
    else:
        print("✗ cyp51A not found")
        gene_found = False
else:
    gene_found = False

## Step 2: DEBUG VCF Content - Sample Analysis

In [ ]:
# DEBUG: Examine actual VCF content to understand format
print("=== DEBUG: EXAMINING VCF CONTENT ===")

# Test with one VCF file first
test_hid = 320  # First file from collection 351

try:
    print(f"\n🔍 Debugging HID {test_hid}...")
    vcf_result = await gxy.get(test_hid)
    vcf_file_path = vcf_result[0] if isinstance(vcf_result, list) else vcf_result
    
    is_compressed = _is_gzipped(vcf_file_path)
    opener = gzip.open if is_compressed else open
    
    print(f"File: {vcf_file_path}")
    print(f"Compressed: {is_compressed}")
    
    # Read first 50 lines to understand format
    with opener(vcf_file_path, 'rt' if is_compressed else 'r') as f:
        lines = []
        for i, line in enumerate(f):
            lines.append(line.strip())
            if i >= 100:  # Read more lines
                break
    
    print(f"\n📋 HEADER ANALYSIS:")
    header_found = False
    data_lines = []
    
    for i, line in enumerate(lines):
        if line.startswith('##'):
            if i < 10:  # Show first few header lines
                print(f"  {line}")
        elif line.startswith('#CHROM'):
            print(f"\n📊 COLUMN HEADER:")
            cols = line.replace('#', '').split('\t')
            for j, col in enumerate(cols):
                print(f"  {j:2d}: {col}")
            header_found = True
        elif not line.startswith('#') and line.strip():
            data_lines.append(line)
            if len(data_lines) >= 10:  # Collect first 10 data lines
                break
    
    if data_lines:
        print(f"\n📊 SAMPLE DATA LINES (first 5):")
        for i, line in enumerate(data_lines[:5], 1):
            fields = line.split('\t')
            print(f"\n  Line {i}: {len(fields)} fields")
            for j, field in enumerate(fields[:10]):  # Show first 10 fields
                print(f"    {j:2d}: {field}")
        
        # Focus on INFO field (usually column 7)
        print(f"\n🔍 INFO FIELD ANALYSIS:")
        for i, line in enumerate(data_lines[:3], 1):
            fields = line.split('\t')
            if len(fields) > 7:
                info_field = fields[7]
                print(f"\n  Line {i} INFO: {info_field}")
                
                # Look for effect annotations
                if 'ANN=' in info_field:
                    print(f"    ✓ Contains ANN= annotation")
                elif 'EFF=' in info_field:
                    print(f"    ✓ Contains EFF= annotation")
                elif 'CSQ=' in info_field:
                    print(f"    ✓ Contains CSQ= annotation")
                else:
                    print(f"    ⚠ No obvious effect annotation found")
                    
                # Check for synonymous terms
                synonymous_terms = ['SYNONYMOUS', 'SILENT', 'synonymous_variant']
                found_synonymous = any(term.lower() in info_field.lower() for term in synonymous_terms)
                print(f"    Synonymous terms found: {found_synonymous}")
    
    print(f"\n📈 VARIANT COUNTING DEBUG:")
    total_variants = 0
    region_variants = 0
    
    # Count variants in target region
    with opener(vcf_file_path, 'rt' if is_compressed else 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            
            total_variants += 1
            fields = line.strip().split('\t')
            
            if len(fields) >= 2:
                chrom = fields[0]
                try:
                    pos = int(fields[1])
                    if gene_found and chrom == chromosome and roi_start <= pos <= roi_end:
                        region_variants += 1
                except ValueError:
                    continue
            
            # Stop after checking reasonable number
            if total_variants >= 50000:
                print(f"    Stopped at {total_variants:,} variants (sampling limit)")
                break
    
    print(f"\n📊 COUNTING RESULTS:")
    print(f"  Total variants scanned: {total_variants:,}")
    print(f"  Variants in target region: {region_variants:,}")
    print(f"  Region hit rate: {region_variants/total_variants*100:.2f}%" if total_variants > 0 else "N/A")
    
except Exception as e:
    print(f"✗ Debug error: {e}")
    import traceback
    traceback.print_exc()

## Step 3: Fixed VCF Processing - Remove Artificial Limits

In [ ]:
# FIXED: VCF processing without artificial limits
print("=== FIXED VCF PROCESSING - NO ARTIFICIAL LIMITS ===")

def debug_synonymous_detection(info_field):
    """Debug version of synonymous detection with logging"""
    if pd.isna(info_field) or not info_field:
        return False, "No INFO field"
    
    info_str = str(info_field).upper()
    
    # Expanded list of synonymous terms
    synonymous_patterns = [
        'SYNONYMOUS_CODING',
        'SYNONYMOUS_VARIANT', 
        'SILENT_MUTATION',
        'SILENT',
        'SYNONYMOUS',
        'synonymous_variant',  # VEP format
        'MODIFIER|SYNONYMOUS',  # SnpEff format
        '|SYNONYMOUS_VARIANT|',  # Another format
    ]
    
    for pattern in synonymous_patterns:
        if pattern.upper() in info_str:
            return True, f"Found: {pattern}"
    
    return False, "No synonymous terms"

def stream_variants_fixed(file_path, target_chromosome, start_pos, end_pos, max_variants=None):
    """FIXED: No artificial 1000-variant limit"""
    is_compressed = _is_gzipped(file_path)
    opener = gzip.open if is_compressed else open
    
    all_region_variants = []
    functional_variants = []
    total_variants = 0
    region_variants = 0
    synonymous_filtered = 0
    
    with opener(file_path, 'rt' if is_compressed else 'r') as f:
        # Find header
        col_names = None
        for line in f:
            if line.startswith('#CHROM'):
                col_names = line.strip().replace('#', '').split('\t')
                break
        
        if col_names is None:
            return [], [], 0, 0, 0
        
        # Process variants
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            
            total_variants += 1
            fields = line.strip().split('\t')
            
            if len(fields) < 8:
                continue
            
            chrom = fields[0]
            try:
                pos = int(fields[1])
            except ValueError:
                continue
            
            # Region filter
            if target_chromosome and chrom == target_chromosome and start_pos <= pos <= end_pos:
                region_variants += 1
                
                # Pad fields to match header
                if len(fields) < len(col_names):
                    fields.extend([''] * (len(col_names) - len(fields)))
                elif len(fields) > len(col_names):
                    fields = fields[:len(col_names)]
                
                all_region_variants.append(fields)
                
                # Check for synonymous (with debugging for first few)
                info_field = fields[7] if len(fields) > 7 else ''
                is_synonymous, reason = debug_synonymous_detection(info_field)
                
                if is_synonymous:
                    synonymous_filtered += 1
                    # Debug: show first few synonymous variants
                    if synonymous_filtered <= 3:
                        print(f"    DEBUG: Synonymous variant #{synonymous_filtered}: {reason}")
                        print(f"           Position: {chrom}:{pos}, INFO: {info_field[:100]}...")
                else:
                    functional_variants.append(fields)
                
                # Apply max limit ONLY if specified and to region variants
                if max_variants and region_variants >= max_variants:
                    print(f"    Reached max region variants limit: {max_variants:,}")
                    break
            
            # Progress for large files
            if total_variants % 100000 == 0:
                print(f"    Progress: {total_variants:,} scanned, {region_variants} in region, {len(functional_variants)} functional")
    
    return all_region_variants, functional_variants, total_variants, region_variants, synonymous_filtered

# Test fixed processing on first 3 files
test_hids = [320, 321, 322]  # First 3 from collection 351

print(f"\nTesting FIXED processing on {len(test_hids)} files...")

for i, hid in enumerate(test_hids, 1):
    try:
        print(f"\n{i}. Processing HID {hid} (FIXED VERSION)...")
        
        vcf_result = await gxy.get(hid)
        vcf_file_path = vcf_result[0] if isinstance(vcf_result, list) else vcf_result
        
        if gene_found:
            # Use high limit but not artificial 1000 cap
            all_region, functional, total, region, synonymous = stream_variants_fixed(
                vcf_file_path, chromosome, roi_start, roi_end, max_variants=10000  # Much higher limit
            )
            
            print(f"   FIXED RESULTS:")
            print(f"     Total variants scanned: {total:,}")
            print(f"     Region variants: {region:,}")
            print(f"     Synonymous filtered: {synonymous:,}")
            print(f"     Functional variants: {len(functional):,}")
            print(f"     Functional rate: {len(functional)/region*100:.1f}%" if region > 0 else "N/A")
            
            # Show what a non-1000 result looks like
            if len(functional) != 1000:
                print(f"     ✅ SUCCESS: No longer exactly 1000 variants!")
            else:
                print(f"     ⚠ Still exactly 1000 - investigate further")
        else:
            print(f"   Skipped (no gene coordinates)")
            
    except Exception as e:
        print(f"   ✗ Error: {e}")

print(f"\n🔍 DEBUG SUMMARY:")
print(f"1. Check if results are no longer exactly 1000 variants")
print(f"2. Check if synonymous filtering is now working (>0 filtered)")
print(f"3. Look for realistic biological variation in results")

## Step 4: Analysis of Results

In [ ]:
# Summary of findings
print("\n" + "="*60)
print("DEBUG ANALYSIS COMPLETE")
print("="*60)

print(f"\n🔍 ISSUES IDENTIFIED:")
print(f"1. ARTIFICIAL LIMIT: Hard-coded 1000-variant cap in stream processing")
print(f"   → FIXED: Removed arbitrary break at 1000 variants")
print(f"   → NOW: Uses realistic limits (10,000) or processes all region variants")

print(f"\n2. SYNONYMOUS FILTERING: May not be working due to INFO field format")
print(f"   → INVESTIGATION: Check if VCF files have effect annotations")
print(f"   → DEBUG: Added detailed synonymous detection logging")
print(f"   → EXPANSION: Added more synonymous term patterns")

print(f"\n🔧 CODE FIXES APPLIED:")
print(f"   ✅ Removed: if len(functional_variants) >= 1000: break")
print(f"   ✅ Added: Detailed INFO field examination")
print(f"   ✅ Added: Synonymous detection debugging")
print(f"   ✅ Added: Progress tracking for large files")
print(f"   ✅ Added: Realistic variant count reporting")

print(f"\n📊 EXPECTED IMPROVEMENTS:")
print(f"   • Variant counts will no longer be exactly 1000")
print(f"   • Synonymous filtering should show >0 filtered (if annotations present)")
print(f"   • Biological variation between samples should be visible")
print(f"   • More realistic functional variant counts")

print(f"\n⚠️ POTENTIAL FINDINGS:")
print(f"   • VCF files may lack effect annotations (SnpEff/VEP)")
print(f"   • Files might be raw variant calls without functional annotation")
print(f"   • May need to implement position-based synonymous detection")

print(f"\n🚀 NEXT STEPS:")
print(f"1. Upload this debug version to Galaxy")
print(f"2. Run and examine actual VCF content")
print(f"3. Create v8.2 with proper functional annotation if needed")
print(f"4. Implement codon-based synonymous detection if VCF lacks annotations")